# Bangladesh: 0237_landsat_lst_sharpening

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0237_landsat_lst_sharpening
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'urban' / '0237_landsat_lst_sharpening'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""00026_landsat_lst_sharpening.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-213: Landsat LST Sharpening using NDVI in Python
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor


"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **Landsat Image Collection**'
landsat = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2025', '2026').filterBounds(roi)
landsat

def ndvi_lst(img):
    qa_band = img.select('QA_PIXEL')
    bit1 = qa_band.bitwiseAnd(1 << 1).neq(0)
    bit2 = qa_band.bitwiseAnd(1 << 2).neq(0)
    bit3 = qa_band.bitwiseAnd(1 << 3).neq(0)
    bit4 = qa_band.bitwiseAnd(1 << 4).neq(0)
    mask = bit1.Or(bit2).Or(bit3).Or(bit4)
    sr_bands = img.select('SR_B.*').multiply(2.75e-05).add(-0.2)
    ndvi = sr_bands.normalizedDifference(['SR_B5', 'SR_B4']).rename('ndvi')
    lst = img.select('ST_B10').multiply(0.00341802).add(149).rename('lst')
    stack = ee.Image.cat([ndvi, lst])
    return stack.updateMask(mask.Not()).copyProperties(img, ['system:time_start'])
landsat_ndvi_lst = landsat.map(ndvi_lst)
landsat_ndvi_lst
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
landsat.first().select('SR_B1').projection()
grid_30m = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:32639', grid_scale=(30, -30))
ds = xr.open_dataset(landsat_ndvi_lst, engine='ee', **grid_30m, chunks='auto')
from dask.diagnostics import ProgressBar
with ProgressBar():
    ds = ds.compute()
'# **Temporal Resampling**'
ds = ds.sortby('time')
ds_monthly = ds.resample(time='M').max('time')
ds_smooth = ds_monthly.rolling(time=3, center=True, min_periods=1).mean('time')
ds_mean = ds_smooth.mean('time')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ds_mean.ndvi.plot(ax=ax[0], robust=True)
ds_mean.lst.plot(ax=ax[1], robust=True)
plt.tight_layout()
'# **LST Sharpening**'
import numpy as np
x = ds_mean.ndvi.values.flatten()
x = x[~np.isnan(x)]
x = x.reshape(-1, 1)
x
y = ds_mean.lst.values.flatten()
y = y[~np.isnan(y)]
y
idx = np.random.choice(len(x), 10000, replace=False)
idx
x_samples = x[idx]
y_samples = y[idx]
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(x_samples, y_samples)
model_slope = model.coef_[0]
ndvi_smooth = ds_mean.ndvi.rolling(x=9, y=9, min_periods=1, center=True).mean('time')
ndvi_details = ds_mean.ndvi - ndvi_smooth
ndvi_details.plot(robust=True)
lst_details = ndvi_details * model_slope
lst_30m = ds_mean.lst + lst_details
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ds_mean.lst.plot(ax=ax[0], robust=True)
lst_30m.plot(ax=ax[1], robust=True)
plt.tight_layout()
xr.corr(ds_mean.lst, lst_30m).item()
from sklearn.metrics import mean_squared_error
np.sqrt(mean_squared_error(ds_mean.lst.values.flatten(), lst_30m.values.flatten()))
import seaborn as sns
sns.regplot(x=ds_mean.lst.values.flatten(), y=lst_30m.values.flatten(), line_kws={'color': 'red'})
dif = ds_mean.lst - lst_30m
dif.plot(robust=True)


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'urban', '0237_landsat_lst_sharpening', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'urban')  # explicit upload; requires Drive OAuth scope
